### CROP DATA - RAW LAYER

######Pre-Requisites
1. DataSource - https://www.data.gov.in/
2. Dataset Link: https://github.com/ankitgilhotra/demo_dbx_user_group/blob/be52f2d1a77580ffee137c66c6f71933abaa06af/crop_production.csv
3. RAW Data Link: https://raw.githubusercontent.com/ankitgilhotra/demo_dbx_user_group/main/crop_production.csv

In [0]:
%skip
import requests

url = "https://raw.githubusercontent.com/ankitgilhotra/demo_dbx_user_group/main/crop_production.csv"

r = requests.get(url)
r.raise_for_status()

with open("/Volumes/dbx_demo/raw/data/crop_production.csv", "wb") as f:
    f.write(r.content)

In [0]:
df = spark.read \
.option("header", True) \
.option("inferSchema", True) \
.csv("/Volumes/dbx_demo/raw/data/crop_production.csv")

row_count = df.count()
print(f"Row count: {row_count}")

display(df.sample(False, 1.0).limit(10))

##### Load Raw Data to bronze layer

### Create table 

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dbx_demo.raw.crop_production (
    State STRING,
    District STRING,
    Crop_Year INT,
    Season STRING,
    Crop STRING,
    Area DOUBLE,
    Production DOUBLE
)
USING DELTA
""")

Load Data to Bronze Layer

In [0]:

# Overwrite table with new data
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbx_demo.raw.crop_production")

In [0]:
display(spark.table("dbx_demo.raw.crop_production").sample(False, 1.0).limit(10))

row_count = spark.sql("SELECT COUNT(*) FROM dbx_demo.raw.crop_production").collect()[0][0]
print(f"Row count: {row_count}")